# Notebook 11: DGE Pathway Analysis

## Overview
This notebook consumes the saved DGE workbooks from notebook 09 and exposes reusable functions for pathway enrichment at either the single-cell-type or whole-level scope.

## Supported Targets
- Single cell type from any DGE workbook sheet
- Entire annotation level: `class`, `subclass`, `supertype`, or `cluster`

## Supported Pathway Libraries
- GO Biological Process
- KEGG
- Reactome
- WikiPathways
- Jensen DISEASES

## Primary Outputs
- Excel workbooks written under `Results/Enrichment/DGE_Pathway_Analysis_nb11/`
- Single-cell-type runs: one workbook named with the resolved cell type, with one tab per requested pathway library
- Whole-level runs: one workbook per cell type in the level, each named with the cell type and level, with one tab per requested pathway library
- Run manifest CSV for provenance and downstream reference

## Usage Pattern
1. Build or refresh the DGE catalog from notebook 09 outputs.
2. Inspect available levels and cell types.
3. Run `run_dge_pathway_analysis(...)` for either a single cell type or a full level.
4. Review the returned result object and the saved Excel workbooks.

In [1]:
from __future__ import annotations

import time
import warnings
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Final, Iterable, Literal

import pandas as pd
from IPython.display import display

warnings.filterwarnings(
    "ignore",
    message=r"urllib3 \(.*\) or chardet \(.*\)/charset_normalizer \(.*\) doesn't match a supported version!",
    category=Warning,
    module="requests",
)

LibraryKey = Literal["go", "kegg", "reactome", "wiki", "jensen"]
Direction = Literal["up", "down", "both"]
TargetKind = Literal["celltype", "level"]


@dataclass(frozen=True)
class Notebook11Config:
    """Central configuration for DGE-driven pathway analysis."""

    analysis_root: Path
    dge_dir: Path
    enrichment_dir: Path
    manifest_path: Path


@dataclass(frozen=True)
class CatalogEntry:
    """One sheet entry from the DGE workbook catalog."""

    level: str
    sheet_name: str
    workbook_path: Path
    normalized_name: str


@dataclass
class PathwayRunResult:
    """Returned object for either cell-type or whole-level enrichment runs."""

    target_kind: TargetKind
    requested_target: str
    resolved_target: str
    output_path: Path
    summary_df: pd.DataFrame
    results_by_sheet: dict[str, pd.DataFrame]


ANALYSIS_ROOT: Final[Path] = Path("/media/drive_c/Project_Brain_snRNAseq/Analysis")
LEVEL_ORDER: Final[tuple[str, ...]] = ("class", "subclass", "supertype", "cluster")
DEFAULT_LIBRARY_ORDER: Final[tuple[LibraryKey, ...]] = (
    "go",
    "kegg",
    "reactome",
    "wiki",
    "jensen",
)

LIBRARY_SPECS: Final[dict[LibraryKey, dict[str, str]]] = {
    "go": {"gene_set": "GO_Biological_Process_2023", "sheet_name": "GO_BP", "label": "GO BP"},
    "kegg": {"gene_set": "KEGG_2016", "sheet_name": "KEGG", "label": "KEGG"},
    "reactome": {"gene_set": "Reactome_2022", "sheet_name": "Reactome", "label": "Reactome"},
    "wiki": {"gene_set": "WikiPathways_2024_Mouse", "sheet_name": "WikiPathways", "label": "WikiPathways"},
    "jensen": {"gene_set": "Jensen_DISEASES", "sheet_name": "Jensen_DISEASES", "label": "Jensen DISEASES"},
}

CONFIG = Notebook11Config(
    analysis_root=ANALYSIS_ROOT,
    dge_dir=ANALYSIS_ROOT / "Results" / "DGE",
    enrichment_dir=ANALYSIS_ROOT / "Results" / "Enrichment",
    manifest_path=ANALYSIS_ROOT / "Results" / "Enrichment" / "pathway_run_manifest_nb11.csv",
)
CONFIG.enrichment_dir.mkdir(parents=True, exist_ok=True)

print("Notebook 11 setup loaded.")
print(f"DGE directory: {CONFIG.dge_dir}")
print(f"Enrichment output directory: {CONFIG.enrichment_dir}")

Notebook 11 setup loaded.
DGE directory: /media/drive_c/Project_Brain_snRNAseq/Analysis/Results/DGE
Enrichment output directory: /media/drive_c/Project_Brain_snRNAseq/Analysis/Results/Enrichment


In [2]:
def normalize_label(text: str) -> str:
    """Normalize labels for robust matching across user input and sheet names."""
    return " ".join(str(text).replace("_", " ").replace("/", " ").lower().split())


def sanitize_name(text: str) -> str:
    """Return a filesystem-safe name fragment."""
    cleaned = [char if char.isalnum() or char in ("-", "_") else "_" for char in str(text)]
    return "".join(cleaned).strip("_")[:120] or "untitled"


def make_excel_sheet_name(text: str, used_names: set[str] | None = None) -> str:
    """Return an Excel-safe sheet name with duplicate protection."""
    banned = set('[]:*?/\\')
    base = "".join("_" if char in banned else char for char in str(text)).strip() or "Sheet"
    base = base[:31]
    if used_names is None:
        return base

    candidate = base
    suffix = 1
    while candidate in used_names:
        suffix_str = f"_{suffix}"
        candidate = f"{base[:31 - len(suffix_str)]}{suffix_str}"
        suffix += 1
    used_names.add(candidate)
    return candidate


def dge_workbook_candidates(level: str) -> list[Path]:
    """Return candidate workbook paths for one DGE level."""
    return [
        CONFIG.dge_dir / f"DGE_level_{level}_nb09.xlsx",
        CONFIG.dge_dir / f"DGE_level_{level}.xlsx",
    ]


def resolve_existing_dge_workbook(level: str) -> Path:
    """Resolve the first existing DGE workbook path for a requested level."""
    for candidate in dge_workbook_candidates(level):
        if candidate.exists():
            return candidate
    tried = "\n".join(str(path) for path in dge_workbook_candidates(level))
    raise FileNotFoundError(f"Missing DGE workbook for level={level}. Tried:\n{tried}")


class DGECatalog:
    """Catalog of notebook 09 DGE workbook sheets by level and cell type."""

    def __init__(self, levels: Iterable[str] = LEVEL_ORDER):
        self.levels = tuple(levels)
        self._entries: list[CatalogEntry] = []
        self._excel_cache: dict[str, pd.ExcelFile] = {}
        self.refresh()

    def refresh(self) -> None:
        """Rebuild the catalog from current DGE workbooks on disk."""
        self._entries = []
        self._excel_cache = {}
        for level in self.levels:
            workbook = resolve_existing_dge_workbook(level)
            excel_file = pd.ExcelFile(workbook)
            self._excel_cache[level] = excel_file
            for sheet_name in excel_file.sheet_names:
                self._entries.append(
                    CatalogEntry(
                        level=level,
                        sheet_name=str(sheet_name),
                        workbook_path=workbook,
                        normalized_name=normalize_label(sheet_name),
                    )
                )

    def get_excel_file(self, level: str) -> pd.ExcelFile:
        """Return the cached Excel handle for one level."""
        if level not in self._excel_cache:
            workbook = resolve_existing_dge_workbook(level)
            self._excel_cache[level] = pd.ExcelFile(workbook)
        return self._excel_cache[level]

    def as_dataframe(self, level: str | None = None) -> pd.DataFrame:
        """Return the catalog as a dataframe for display or filtering."""
        rows = [
            {
                "level": entry.level,
                "cell_type": entry.sheet_name,
                "workbook": str(entry.workbook_path),
            }
            for entry in self._entries
            if level is None or entry.level == level
        ]
        return pd.DataFrame(rows)

    def list_levels(self) -> list[str]:
        """Return levels that currently have workbook-backed entries."""
        return sorted({entry.level for entry in self._entries}, key=lambda item: self.levels.index(item))

    def list_cell_types(self, level: str | None = None) -> list[str]:
        """Return cell type names, optionally restricted to one level."""
        return sorted(entry.sheet_name for entry in self._entries if level is None or entry.level == level)

    def resolve_celltype(self, query: str, level: str | None = None) -> CatalogEntry:
        """Resolve a user query to one catalog entry, with ambiguity guards."""
        normalized_query = normalize_label(query)
        candidates = [entry for entry in self._entries if level is None or entry.level == level]

        exact = [entry for entry in candidates if entry.normalized_name == normalized_query]
        if len(exact) == 1:
            return exact[0]
        if len(exact) > 1 and level is None:
            levels_found = sorted({entry.level for entry in exact})
            raise ValueError(
                f"Cell type '{query}' matches multiple levels {levels_found}. Pass level= to disambiguate."
            )

        contains = [entry for entry in candidates if normalized_query in entry.normalized_name]
        if len(contains) == 1:
            return contains[0]
        if len(contains) > 1:
            preview = [f"{entry.level}: {entry.sheet_name}" for entry in contains[:10]]
            raise ValueError(
                f"Cell type '{query}' matched multiple entries: {preview}. Use a more specific name or pass level=."
            )

        raise ValueError(
            f"Cell type '{query}' was not found in the DGE catalog. Use catalog.as_dataframe() to inspect valid names."
        )

    def get_level_entries(self, level: str) -> list[CatalogEntry]:
        """Return all catalog entries for one level."""
        if level not in self.list_levels():
            raise ValueError(f"Level '{level}' is not available. Valid levels: {self.list_levels()}")
        return [entry for entry in self._entries if entry.level == level]


def infer_first_existing(columns: Iterable[str], candidates: Iterable[str], label: str) -> str:
    """Return the first matching column name from a list of candidates."""
    column_list = list(columns)
    for candidate in candidates:
        if candidate in column_list:
            return candidate
    raise KeyError(f"Could not identify {label} column. Available columns: {column_list}")


def read_dge_sheet(catalog: DGECatalog, entry: CatalogEntry) -> pd.DataFrame:
    """Load one DGE sheet and normalize the key columns used for enrichment."""
    df = pd.read_excel(catalog.get_excel_file(entry.level), sheet_name=entry.sheet_name)
    gene_col = infer_first_existing(df.columns, ["gene", "Gene", "names", "gene_symbol"], "gene")
    logfc_col = infer_first_existing(
        df.columns,
        ["logFC", "avg_log2FC", "log2FoldChange", "logfoldchanges", "avg_logFC"],
        "logFC",
    )
    pval_col = infer_first_existing(
        df.columns,
        ["p_adj", "p_adj_scanpy", "pvals_adj", "adj.P.Val", "FDR", "p_val_adj", "p_val"],
        "adjusted p-value",
    )

    work = df.copy()
    work["gene"] = work[gene_col].astype(str)
    work["logFC"] = pd.to_numeric(work[logfc_col], errors="coerce")
    work["p_adj_used"] = pd.to_numeric(work[pval_col], errors="coerce")
    work = work.dropna(subset=["gene", "logFC", "p_adj_used"]).copy()
    return work

In [3]:
def resolve_library_selection(libraries: str | Iterable[str] = "all") -> tuple[LibraryKey, ...]:
    """Normalize user library selections into canonical library keys."""
    if isinstance(libraries, str):
        if libraries.strip().lower() == "all":
            return DEFAULT_LIBRARY_ORDER
        requested = [item.strip().lower() for item in libraries.split(",") if item.strip()]
    else:
        requested = [str(item).strip().lower() for item in libraries]

    alias_map = {
        "go": "go",
        "go_bp": "go",
        "gobp": "go",
        "kegg": "kegg",
        "reactome": "reactome",
        "wiki": "wiki",
        "wikipathways": "wiki",
        "jensen": "jensen",
        "jensen_diseases": "jensen",
        "janseon": "jensen",
    }

    resolved: list[LibraryKey] = []
    for item in requested:
        if item not in alias_map:
            valid = sorted(alias_map)
            raise ValueError(f"Unknown library selection '{item}'. Valid options include: {valid}")
        key = alias_map[item]
        if key not in resolved:
            resolved.append(key)

    if not resolved:
        raise ValueError("No valid libraries were selected.")
    return tuple(resolved)


def select_enrichment_genes(
    dge_df: pd.DataFrame,
    direction: Direction = "both",
    p_adj_threshold: float = 0.05,
    min_abs_logfc: float = 0.25,
    max_genes: int = 300,
) -> dict[str, list[str]]:
    """Select enrichment-ready genes from one DGE table by direction."""
    filtered = dge_df[
        (dge_df["p_adj_used"] <= p_adj_threshold)
        & (dge_df["logFC"].abs() >= min_abs_logfc)
    ].copy()

    if filtered.empty:
        return {"up": [], "down": []}

    filtered = filtered.sort_values(["p_adj_used", "logFC"], ascending=[True, False])
    up_genes = filtered.loc[filtered["logFC"] > 0, "gene"].drop_duplicates().head(max_genes).tolist()
    down_genes = filtered.loc[filtered["logFC"] < 0, "gene"].drop_duplicates().head(max_genes).tolist()

    if direction == "up":
        return {"up": up_genes, "down": []}
    if direction == "down":
        return {"up": [], "down": down_genes}
    return {"up": up_genes, "down": down_genes}


def run_enrichr_with_retry(
    genes: list[str],
    gene_set_name: str,
    organism: str = "mouse",
    max_retries: int = 4,
    retry_base_seconds: float = 2.0,
) -> pd.DataFrame:
    """Run Enrichr with retry/backoff and return a dataframe."""
    if not genes:
        return pd.DataFrame()

    try:
        import gseapy as gp
    except ImportError as exc:
        raise ImportError(
            "gseapy is required for notebook 11 enrichment runs. Install it in the active kernel before executing analysis."
        ) from exc

    last_error: Exception | None = None
    for attempt in range(max_retries + 1):
        try:
            enr = gp.enrichr(
                gene_list=genes,
                gene_sets=gene_set_name,
                organism=organism,
                outdir=None,
                cutoff=1.0,
                no_plot=True,
            )
            result_df = getattr(enr, "results", pd.DataFrame())
            return result_df.copy() if isinstance(result_df, pd.DataFrame) else pd.DataFrame()
        except Exception as exc:
            last_error = exc
            message = str(exc).lower()
            should_retry = (
                "429" in message
                or "too many requests" in message
                or "rate" in message
            ) and attempt < max_retries
            if not should_retry:
                break
            time.sleep(retry_base_seconds * (2 ** attempt))

    raise RuntimeError(f"Enrichr failed for gene set {gene_set_name}: {last_error}")


def run_library_enrichment(
    dge_df: pd.DataFrame,
    library_keys: tuple[LibraryKey, ...],
    direction: Direction = "both",
    p_adj_threshold: float = 0.05,
    min_abs_logfc: float = 0.25,
    max_genes: int = 300,
    fdr_threshold: float = 0.05,
    top_n_per_library: int = 25,
    organism: str = "mouse",
) -> pd.DataFrame:
    """Run enrichment across one DGE dataframe for the requested pathway libraries."""
    gene_sets = select_enrichment_genes(
        dge_df=dge_df,
        direction=direction,
        p_adj_threshold=p_adj_threshold,
        min_abs_logfc=min_abs_logfc,
        max_genes=max_genes,
    )

    result_frames: list[pd.DataFrame] = []
    for direction_key, genes in gene_sets.items():
        if not genes:
            continue
        for library_key in library_keys:
            spec = LIBRARY_SPECS[library_key]
            library_df = run_enrichr_with_retry(
                genes=genes,
                gene_set_name=spec["gene_set"],
                organism=organism,
            )
            if library_df.empty or "Adjusted P-value" not in library_df.columns:
                continue

            work = library_df.copy()
            work["Adjusted P-value"] = pd.to_numeric(work["Adjusted P-value"], errors="coerce")
            work = work.dropna(subset=["Adjusted P-value"]).copy()
            work = work[work["Adjusted P-value"] <= fdr_threshold].copy()
            if work.empty:
                continue

            work = work.sort_values("Adjusted P-value", ascending=True).head(top_n_per_library)
            work.insert(0, "library_key", library_key)
            work.insert(1, "library_label", spec["label"])
            work.insert(2, "direction", direction_key)
            work.insert(3, "n_input_genes", len(genes))
            result_frames.append(work)

    if not result_frames:
        return pd.DataFrame(
            columns=[
                "library_key",
                "library_label",
                "direction",
                "n_input_genes",
                "Term",
                "Adjusted P-value",
            ]
        )

    return pd.concat(result_frames, ignore_index=True).sort_values(
        ["library_key", "direction", "Adjusted P-value"],
        ascending=[True, True, True],
    ).reset_index(drop=True)


def append_run_manifest(
    target_kind: TargetKind,
    requested_target: str,
    resolved_target: str,
    output_path: Path,
    library_keys: tuple[LibraryKey, ...],
    direction: Direction,
    summary_df: pd.DataFrame,
) -> None:
    """Append one run record to the notebook 11 manifest CSV."""
    manifest_row = pd.DataFrame(
        [
            {
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "target_kind": target_kind,
                "requested_target": requested_target,
                "resolved_target": resolved_target,
                "libraries": "|".join(library_keys),
                "direction": direction,
                "output_path": str(output_path),
                "summary_rows": int(len(summary_df)),
            }
        ]
    )

    if CONFIG.manifest_path.exists():
        previous = pd.read_csv(CONFIG.manifest_path)
        pd.concat([previous, manifest_row], ignore_index=True).to_csv(CONFIG.manifest_path, index=False)
    else:
        CONFIG.manifest_path.parent.mkdir(parents=True, exist_ok=True)
        manifest_row.to_csv(CONFIG.manifest_path, index=False)


def write_celltype_workbook(
    entry: CatalogEntry,
    combined_df: pd.DataFrame,
    library_keys: tuple[LibraryKey, ...],
    summary_df: pd.DataFrame,
) -> Path:
    """Write one workbook for a single resolved cell type, one tab per library."""
    output_path = CONFIG.enrichment_dir / (
        f"{sanitize_name(entry.sheet_name)}__{entry.level}__pathway_nb11.xlsx"
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(output_path) as writer:
        summary_df.to_excel(writer, sheet_name="run_summary", index=False)
        for library_key in library_keys:
            sheet_name = LIBRARY_SPECS[library_key]["sheet_name"]
            subset = combined_df[combined_df["library_key"] == library_key].copy()
            if subset.empty:
                subset = pd.DataFrame(
                    [{"status": "no_significant_results", "library_key": library_key, "library_label": LIBRARY_SPECS[library_key]["label"]}]
                )
            subset.to_excel(writer, sheet_name=sheet_name[:31], index=False)

    return output_path


def build_summary_row(entry: CatalogEntry, result_df: pd.DataFrame) -> dict[str, object]:
    """Build a compact run summary row for a cell type or sheet result."""
    if result_df.empty:
        return {
            "level": entry.level,
            "cell_type": entry.sheet_name,
            "n_rows": 0,
            "libraries_returned": "",
            "best_adjusted_p": None,
        }

    return {
        "level": entry.level,
        "cell_type": entry.sheet_name,
        "n_rows": int(len(result_df)),
        "libraries_returned": "|".join(sorted(result_df["library_key"].dropna().astype(str).unique())),
        "best_adjusted_p": float(result_df["Adjusted P-value"].min()),
    }


def run_dge_pathway_analysis(
    target: str,
    target_kind: TargetKind = "celltype",
    level: str | None = None,
    libraries: str | Iterable[str] = "all",
    direction: Direction = "both",
    p_adj_threshold: float = 0.05,
    min_abs_logfc: float = 0.25,
    max_genes: int = 300,
    fdr_threshold: float = 0.05,
    top_n_per_library: int = 25,
    organism: str = "mouse",
    catalog: DGECatalog | None = None,
) -> PathwayRunResult:
    """Public notebook 11 entrypoint for DGE-driven pathway analysis and Excel export."""
    dge_catalog = catalog or DGECatalog()
    library_keys = resolve_library_selection(libraries)

    if target_kind == "celltype":
        entry = dge_catalog.resolve_celltype(query=target, level=level)
        dge_df = read_dge_sheet(dge_catalog, entry)
        combined_df = run_library_enrichment(
            dge_df=dge_df,
            library_keys=library_keys,
            direction=direction,
            p_adj_threshold=p_adj_threshold,
            min_abs_logfc=min_abs_logfc,
            max_genes=max_genes,
            fdr_threshold=fdr_threshold,
            top_n_per_library=top_n_per_library,
            organism=organism,
        )
        summary_df = pd.DataFrame([build_summary_row(entry, combined_df)])
        output_path = write_celltype_workbook(
            entry=entry,
            combined_df=combined_df,
            library_keys=library_keys,
            summary_df=summary_df,
        )
        append_run_manifest(
            target_kind="celltype",
            requested_target=target,
            resolved_target=f"{entry.level}:{entry.sheet_name}",
            output_path=output_path,
            library_keys=library_keys,
            direction=direction,
            summary_df=summary_df,
        )
        return PathwayRunResult(
            target_kind="celltype",
            requested_target=target,
            resolved_target=f"{entry.level}:{entry.sheet_name}",
            output_path=output_path,
            summary_df=summary_df,
            results_by_sheet={entry.sheet_name: combined_df},
        )

    level_name = (level or target).strip().lower()
    if level_name not in LEVEL_ORDER:
        raise ValueError(f"For target_kind='level', target must be one of {LEVEL_ORDER}.")

    entries = dge_catalog.get_level_entries(level_name)
    per_sheet_results: dict[str, pd.DataFrame] = {}
    summary_rows: list[dict[str, object]] = []
    output_paths: list[Path] = []

    for entry in entries:
        dge_df = read_dge_sheet(dge_catalog, entry)
        combined_df = run_library_enrichment(
            dge_df=dge_df,
            library_keys=library_keys,
            direction=direction,
            p_adj_threshold=p_adj_threshold,
            min_abs_logfc=min_abs_logfc,
            max_genes=max_genes,
            fdr_threshold=fdr_threshold,
            top_n_per_library=top_n_per_library,
            organism=organism,
        )
        per_sheet_results[entry.sheet_name] = combined_df
        summary_rows.append(build_summary_row(entry, combined_df))
        
        # Write individual workbook for this cell type
        single_cell_summary = pd.DataFrame([build_summary_row(entry, combined_df)])
        output_path = write_celltype_workbook(
            entry=entry,
            combined_df=combined_df,
            library_keys=library_keys,
            summary_df=single_cell_summary,
        )
        output_paths.append(output_path)
        append_run_manifest(
            target_kind="level_celltype",
            requested_target=target,
            resolved_target=f"{entry.level}:{entry.sheet_name}",
            output_path=output_path,
            library_keys=library_keys,
            direction=direction,
            summary_df=single_cell_summary,
        )

    # Create summary of all cell types processed for the level
    summary_df = pd.DataFrame(summary_rows).sort_values(["n_rows", "cell_type"], ascending=[False, True])
    primary_output = output_paths[0] if output_paths else CONFIG.enrichment_dir / f"level_{level_name}_nb11.xlsx"
    
    return PathwayRunResult(
        target_kind="level",
        requested_target=target,
        resolved_target=level_name,
        output_path=primary_output,
        summary_df=summary_df,
        results_by_sheet=per_sheet_results,
    )

## Runner and Examples

The catalog and runner below are the main notebook 11 interfaces.

- Use `catalog.as_dataframe()` to inspect available DGE-backed cell types and levels.
- Use `run_dge_pathway_analysis(target=..., target_kind="celltype")` for one cell type.
- Use `run_dge_pathway_analysis(target="class", target_kind="level")` for a full level.
- `libraries` accepts `"all"` or multiple selections such as `["go", "reactome", "jensen"]`.
- Single-cell-type exports write one Excel workbook with one tab per requested pathway library.
- Whole-level exports produce one Excel workbook per cell type in the level, each with one tab per requested pathway library.

In [4]:
catalog = DGECatalog()

print("Available levels:", catalog.list_levels())
display(catalog.as_dataframe().head(12))

RUN_ANALYSIS = True
RUN_TARGET_KIND: TargetKind = "level"
RUN_TARGET = "class"
RUN_LEVEL: str | None = None
RUN_LIBRARIES: str | list[str] = ["go", "kegg", "reactome", "wiki", "jensen"]
RUN_DIRECTION: Direction = "both"

if RUN_ANALYSIS:
    run_result = run_dge_pathway_analysis(
        target=RUN_TARGET,
        target_kind=RUN_TARGET_KIND,
        level=RUN_LEVEL,
        libraries=RUN_LIBRARIES,
        direction=RUN_DIRECTION,
        p_adj_threshold=0.05,
        min_abs_logfc=0.25,
        max_genes=300,
        fdr_threshold=0.05,
        top_n_per_library=25,
        organism="mouse",
        catalog=catalog,
    )
    print(f"Saved workbook: {run_result.output_path}")
    display(run_result.summary_df)
    for sheet_name, result_df in run_result.results_by_sheet.items():
        print(f"Preview: {sheet_name}")
        display(result_df.head(10))
        break
else:
    print(
        "RUN_ANALYSIS is False. Set it to True and rerun this cell to execute notebook 11 pathway analysis."
    )
    print(
        "Examples: run_dge_pathway_analysis(target='class', target_kind='level', libraries='all') "
        "or run_dge_pathway_analysis(target='Astro', target_kind='celltype', libraries=['go', 'reactome'])."
    )

Available levels: ['class', 'subclass', 'supertype', 'cluster']


,level,cell_type,workbook
0,class,01 IT-ET Glut,/media/drive_c/Project_Brain_snRNAseq/Analysis...
1,class,02 NP-CT-L6b Glut,/media/drive_c/Project_Brain_snRNAseq/Analysis...
2,class,03 OB-CR Glut,/media/drive_c/Project_Brain_snRNAseq/Analysis...
3,class,04 DG-IMN Glut,/media/drive_c/Project_Brain_snRNAseq/Analysis...
4,class,05 OB-IMN GABA,/media/drive_c/Project_Brain_snRNAseq/Analysis...
5,class,06 CTX-CGE GABA,/media/drive_c/Project_Brain_snRNAseq/Analysis...
6,class,07 CTX-MGE GABA,/media/drive_c/Project_Brain_snRNAseq/Analysis...
7,class,08 CNU-MGE GABA,/media/drive_c/Project_Brain_snRNAseq/Analysis...
8,class,09 CNU-LGE GABA,/media/drive_c/Project_Brain_snRNAseq/Analysis...
9,class,10 LSX GABA,/media/drive_c/Project_Brain_snRNAseq/Analysis...


Saved workbook: /media/drive_c/Project_Brain_snRNAseq/Analysis/Results/Enrichment/01_IT-ET_Glut__class__pathway_nb11.xlsx


,level,cell_type,n_rows,libraries_returned,best_adjusted_p
27,class,33 Vascular,145,go|jensen|kegg|reactome|wiki,2.026375e-08
11,class,12 HY GABA,144,go|jensen|kegg|reactome|wiki,3.322702e-19
14,class,18 TH Glut,122,go|jensen|kegg|reactome|wiki,1.155505e-17
24,class,30 Astro-Epen,109,go|jensen|kegg|reactome|wiki,2.935469e-05
8,class,09 CNU-LGE GABA,107,go|jensen|kegg|reactome|wiki,4.132307e-08
15,class,19 MB Glut,96,go|jensen|kegg|reactome|wiki,1.260585e-08
28,class,34 Immune,92,go|jensen|kegg|reactome|wiki,4.496263e-06
3,class,04 DG-IMN Glut,80,go|jensen|kegg|reactome|wiki,1.816946e-07
4,class,05 OB-IMN GABA,80,go|jensen|kegg|reactome|wiki,1.597074e-11
7,class,08 CNU-MGE GABA,78,go|jensen|kegg|reactome|wiki,2.260009e-07


Preview: 01 IT-ET Glut


,library_key,library_label,direction,n_input_genes,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes
0,go,GO BP,down,300,GO_Biological_Process_2023,Negative Regulation Of Nucleic Acid-Templated ...,20/456,0.000020,0.026633,0,0,3.155963,34.122226,KLF10;HSPA8;HDAC5;CIPC;NONO;BHLHE41;NR1D2;NR1D...
1,go,GO BP,down,300,GO_Biological_Process_2023,Organic Hydroxy Compound Biosynthetic Process ...,6/42,0.000036,0.026633,0,0,11.147392,114.034199,PER2;SQLE;OSBPL2;PNPO;MSMO1;HMGCR
2,go,GO BP,down,300,GO_Biological_Process_2023,Regulation Of Protein Ubiquitination (GO:0031396),9/113,0.000052,0.026633,0,0,5.827518,57.479178,DNAJA1;PER2;GSK3B;LRRK2;CTNNB1;SPRY2;PRICKLE1;...
3,go,GO BP,down,300,GO_Biological_Process_2023,Regulation Of Neuron Projection Development (G...,11/174,0.000066,0.026633,0,0,4.562103,43.883845,RTN4R;SETX;GSK3B;NTRK2;FUT9;LRRK2;CFL1;SERPINI...
4,go,GO BP,down,300,GO_Biological_Process_2023,Positive Regulation Of Cell-Substrate Adhesion...,7/71,0.000093,0.026633,0,0,7.329991,68.026373,CDC42;GSK3B;ENC1;MYADM;RELL2;CALR;CX3CL1
5,go,GO BP,down,300,GO_Biological_Process_2023,Negative Regulation Of Microglial Cell Activat...,3/7,0.000112,0.026633,0,0,49.737374,452.532704,SYT11;NR1D1;CX3CL1
6,go,GO BP,down,300,GO_Biological_Process_2023,Cellular Response To Growth Factor Stimulus (G...,10/155,0.000120,0.026633,0,0,4.650416,41.980602,MAP2K3;SETX;NR4A1;EGR3;BMPR2;DUSP3;CTNNB1;SPRY...
7,go,GO BP,down,300,GO_Biological_Process_2023,Proteasomal Protein Catabolic Process (GO:0010...,12/220,0.000129,0.026633,0,0,3.904647,34.973848,APC2;GSK3B;PSMC6;PSMD11;PSMA1;ENC1;SYVN1;CTNNB...
8,go,GO BP,down,300,GO_Biological_Process_2023,Positive Regulation Of Neurogenesis (GO:0050769),7/75,0.000132,0.026633,0,0,6.897410,61.601660,NTRK2;NUMBL;FZD3;CTNNB1;SOX8;CX3CL1;PLXNA3
9,go,GO BP,down,300,GO_Biological_Process_2023,Regulation Of Regulated Secretory Pathway (GO:...,5/35,0.000165,0.029886,0,0,11.112994,96.799770,SYT4;RIMS4;PREPL;SYT11;LRRK2
